# Proyecto 1
Nombre: Johan Sebastian Camacho Tuay     
Fecha: 04/09/2026

In [1]:
#importar las clases
import sys
from goodrich.ch06.array_stack import*
from goodrich.ch06.array_queue import*

## 1. Clase DataProcessor
Errores usados(se mantienen igual en toda la clase):

- ValueError: el registro no tiene tres componentes o su valor no es numerico.
- Empty: no hay registros pendientes, o no hay cambios para deshacer/rehacer.
- KeyError: se consulta un sensor/variable que nunca fue procesado.

In [2]:
class DataProcessor:
    """Sistema para recibir y procesar registros de datos.
    Errores que produce la clase:
      - ValueError : el registro no tiene tres componentes o su valor no es numerico.
      - Empty      : no hay registros pendientes, o no hay cambios para deshacer/rehacer.
      - KeyError   : se consulta un sensor/variable que nunca fue procesado.
    """

    def __init__(self):
        self._pendientes = ArrayQueue()   # registros que llegaron y aun no se procesan
        self._historial = ArrayStack()    # cambios ya hechos, para poder deshacerlos
        self._estado = []                 # como estan los registros actualmente
        self._rehacer = ArrayStack()      # bonus: cambios deshechos que se pueden repetir

    # ---------- metodos auxiliares ----------

    def _buscar(self, sensor, variable):
        """Devuelve la posicion del dato en el estado actual, o -1 si no existe."""
        for i in range(len(self._estado)):
            if self._estado[i][0] == sensor and self._estado[i][1] == variable:
                return i
        return -1

    def state(self):
        """Copia del estado actual. Solo se usa en las celdas de prueba."""
        return list(self._estado)

    # ---------- 3. add ----------

    def add(self, record):
        if not isinstance(record, tuple):
            raise ValueError('El registro debe ser una tupla')
        if len(record) != 3:
            raise ValueError('El registro debe tener exactamente tres componentes')
        valor = record[2]
        if not isinstance(valor, (int, float)):
            raise ValueError('El valor del registro debe ser numerico')
        self._pendientes.enqueue(record)

    # ---------- 4. process_next ----------

    def process_next(self):
        if self._pendientes.is_empty():
            raise Empty('No hay registros pendientes')

        registro = self._pendientes.dequeue()
        sensor = registro[0]
        variable = registro[1]
        i = self._buscar(sensor, variable)

        if i == -1:
            # el dato no existia: el cambio fue una creacion
            cambio = (registro, False, None)
            self._estado.append(registro)
        else:
            # el dato ya existia: se guarda el valor anterior
            valor_anterior = self._estado[i][2]
            cambio = (registro, True, valor_anterior)
            self._estado[i] = registro

        self._historial.push(cambio)
        self._rehacer = ArrayStack()   # un cambio nuevo anula los redo pendientes
        return registro

    # ---------- 5. undo ----------

    def undo(self):
        if self._historial.is_empty():
            raise Empty('No hay cambios para deshacer')

        cambio = self._historial.pop()
        registro = cambio[0]
        existia = cambio[1]
        valor_anterior = cambio[2]

        i = self._buscar(registro[0], registro[1])
        if existia:
            self._estado[i] = (registro[0], registro[1], valor_anterior)
        else:
            self._estado.pop(i)

        self._rehacer.push(cambio)

    # ---------- 6. pending ----------

    def pending(self):
        return len(self._pendientes)

    # ---------- 7. current_value ----------

    def current_value(self, sensor, variable):
        i = self._buscar(sensor, variable)
        if i == -1:
            raise KeyError('No existe un valor para ese sensor y variable')
        return self._estado[i][2]

    # ---------- 12. bonus: redo ----------

    def redo(self):
        if self._rehacer.is_empty():
            raise Empty('No hay cambios para rehacer')

        cambio = self._rehacer.pop()
        registro = cambio[0]

        i = self._buscar(registro[0], registro[1])
        if i == -1:
            self._estado.append(registro)
        else:
            self._estado[i] = registro

        self._historial.push(cambio)
        return registro

## 2. Ejemplo completo del enunciado

In [3]:
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
print("pendientes:", p.pending())

print("procesa:", p.process_next(), "-> estado:", p.state())
print("procesa:", p.process_next(), "-> estado:", p.state())
print("procesa:", p.process_next(), "-> estado:", p.state())

p.undo()
print("undo -> estado:", p.state())
p.undo()
print("undo -> estado:", p.state())
p.undo()
print("undo -> estado:", p.state())

pendientes: 3
procesa: ('S01', 'temperature', 20) -> estado: [('S01', 'temperature', 20)]
procesa: ('S01', 'temperature', 25) -> estado: [('S01', 'temperature', 25)]
procesa: ('S01', 'humidity', 60) -> estado: [('S01', 'temperature', 25), ('S01', 'humidity', 60)]
undo -> estado: [('S01', 'temperature', 25)]
undo -> estado: [('S01', 'temperature', 20)]
undo -> estado: []


## 3. Pruebas obligatorias

### 3.1 pending() sobre un procesador vacio

In [4]:
p = DataProcessor()
print("pendientes:", p.pending())

pendientes: 0


### 3.2 Agregar un registro

In [5]:
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
print("pendientes:", p.pending())

pendientes: 1


### 3.3 Agregar varios registros

In [6]:
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.add(("S02", "temperature", 25.1))
p.add(("S01", "humidity", 61.2))
print("pendientes:", p.pending())

pendientes: 3


### 3.4 Verificar procesamiento FIFO

In [7]:
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)

print("primero:", p.process_next())
print("segundo:", p.process_next())
print("tercero:", p.process_next())

primero: ('S01', 'temperature', 20)
segundo: ('S01', 'temperature', 25)
tercero: ('S01', 'humidity', 60)


### 3.5 Procesar un registro

In [8]:
p = DataProcessor()
p.add(("S01", "temperature", 23.5))

print("procesado:", p.process_next())
print("estado:", p.state())
print("pendientes:", p.pending())

procesado: ('S01', 'temperature', 23.5)
estado: [('S01', 'temperature', 23.5)]
pendientes: 0


### 3.6 Procesar varios registros

In [9]:
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.add(("S02", "temperature", 25.1))
p.add(("S01", "humidity", 61.2))

while p.pending() > 0:
    print("procesado:", p.process_next())

print("estado:", p.state())

procesado: ('S01', 'temperature', 23.5)
procesado: ('S02', 'temperature', 25.1)
procesado: ('S01', 'humidity', 61.2)
estado: [('S01', 'temperature', 23.5), ('S02', 'temperature', 25.1), ('S01', 'humidity', 61.2)]


### 3.7 Actualizar una variable existente

In [10]:
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.add(("S01", "temperature", 27.0))
p.process_next()
print("despues del primero:", p.state())
p.process_next()
print("despues del segundo:", p.state())
print("cantidad de datos en el estado:", len(p.state()))

despues del primero: [('S01', 'temperature', 20)]
despues del segundo: [('S01', 'temperature', 27.0)]
cantidad de datos en el estado: 1


### 3.8 Consultar el valor actual

In [11]:
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.process_next()
print("valor actual:", p.current_value("S01", "temperature"))

valor actual: 23.5


### 3.9 Realizar un `undo()`

In [12]:
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.add(("S01", "temperature", 25))
p.process_next()
p.process_next()
print("antes del undo:", p.current_value("S01", "temperature"))

p.undo()
print("despues del undo:", p.current_value("S01", "temperature"))

antes del undo: 25
despues del undo: 20


### 3.10 Realizar varios `undo()` consecutivos

In [13]:
p = DataProcessor()
for valor in [20, 25, 30]:
    p.add(("S01", "temperature", valor))
    p.process_next()

print("estado inicial:", p.current_value("S01", "temperature"))
p.undo()
print("undo 1:", p.current_value("S01", "temperature"))
p.undo()
print("undo 2:", p.current_value("S01", "temperature"))

estado inicial: 30
undo 1: 25
undo 2: 20


### 3.11 Procesar cuando la Queue esta vacia

In [14]:
p = DataProcessor()
try:
    p.process_next()
except Empty as e:
    print("Empty:", e)

Empty: No hay registros pendientes


### 3.12 `undo()` cuando el historial esta vacio

In [15]:
p = DataProcessor()
try:
    p.undo()
except Empty as e:
    print("Empty:", e)

Empty: No hay cambios para deshacer


### 3.13 Deshacer la creacion de un dato que antes no existia

In [16]:
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.process_next()
print("estado despues de procesar:", p.state())

p.undo()
print("estado despues del undo:", p.state())

try:
    p.current_value("S01", "temperature")
except KeyError as e:
    print("KeyError:", e)

estado despues de procesar: [('S01', 'temperature', 23.5)]
estado despues del undo: []
KeyError: 'No existe un valor para ese sensor y variable'


### 3.14 Varios cambios sobre la misma variable

In [17]:
p = DataProcessor()
for valor in [20, 25, 30, 35]:
    p.add(("S01", "temperature", valor))
    p.process_next()
    print("valor actual:", p.current_value("S01", "temperature"))

print("el estado sigue teniendo un solo dato:", p.state())

valor actual: 20
valor actual: 25
valor actual: 30
valor actual: 35
el estado sigue teniendo un solo dato: [('S01', 'temperature', 35)]


### 3.15 Agregar un registro con formato incorrecto

In [18]:
p = DataProcessor()
try:
    p.add(("S01", 23.5))
except ValueError as e:
    print("ValueError:", e)

try:
    p.add(("S01", "temperature", 23.5, "extra"))
except ValueError as e:
    print("ValueError:", e)

ValueError: El registro debe tener exactamente tres componentes
ValueError: El registro debe tener exactamente tres componentes


### 3.16 Agregar un registro cuyo valor no sea numerico

In [19]:
p = DataProcessor()
try:
    p.add(("S01", "temperature", "23.5"))
except ValueError as e:
    print("ValueError:", e)

ValueError: El valor del registro debe ser numerico


### 3.17 Consultar un sensor/variable que nunca fue procesado

In [20]:
p = DataProcessor()
try:
    p.current_value("S99", "presion")
except KeyError as e:
    print("KeyError:", e)

KeyError: 'No existe un valor para ese sensor y variable'


## 4. Decisiones de diseno

**Que se guarda en el Stack.** Para poder deshacer *exactamente* un cambio no basta con
guardar el registro procesado: hay que saber como estaba el dato antes. Por eso en el
historial se guarda una tupla de tres partes:

```
(registro_procesado, existia_antes, valor_anterior)
```

- Si el dato **no existia**, se guarda `(registro, False, None)` y el `undo()` lo elimina
  del estado.
- Si el dato **ya existia**, se guarda `(registro, True, valor_anterior)` y el `undo()`
  devuelve el valor viejo.

Asi un solo `undo()` sabe que hacer sin tener que recorrer todo el historial.

**Por que una lista para el estado.** El enunciado pide explicitamente un `list` para el
estado actual. Como cada elemento es la tupla completa `(sensor, variable, value)`, para
encontrar un dato hay que recorrer la lista comparando las dos primeras posiciones. Eso se
concentro en el metodo auxiliar `_buscar`, que devuelve la posicion o `-1`, para no repetir
el mismo `for` en `process_next`, `undo`, `current_value` y `redo`.

**Actualizar en vez de agregar.** Cuando llega un registro de una pareja
`(sensor, variable)` que ya existe, se reemplaza la tupla en su posicion en lugar de
agregar una nueva. De esa forma la lista nunca tiene dos entradas para el mismo dato y
`current_value` siempre devuelve el ultimo valor procesado.

**Validaciones en `add` y no en `process_next`.** Un registro invalido se rechaza apenas
llega, para que la cola solo contenga registros que se pueden procesar. Se uso `ValueError`
en los dos casos (tamano incorrecto y valor no numerico) para mantener un solo tipo de
error de validacion.

**Errores.** `Empty` se usa tal como viene del repositorio del curso, tanto para la cola
vacia en `process_next` como para el historial vacio en `undo`. `KeyError` se usa en
`current_value` porque se esta consultando por una clave que no existe.

**La clase no imprime.** Todos los metodos devuelven valores o producen errores; los `print`
solo aparecen en las celdas de prueba, como lo pide el enunciado.

### 6.1 Prueba de `redo()`

In [21]:
p = DataProcessor()
for valor in [20, 25, 30]:
    p.add(("S01", "temperature", valor))
    p.process_next()

print("estado:", p.current_value("S01", "temperature"))
p.undo()
print("undo :", p.current_value("S01", "temperature"))
p.redo()
print("redo :", p.current_value("S01", "temperature"))

estado: 30
undo : 25
redo : 30


### 6.2 Varios `undo()` y varios `redo()`

In [22]:
p = DataProcessor()
for valor in [20, 25, 30]:
    p.add(("S01", "temperature", valor))
    p.process_next()

p.undo()
p.undo()
print("despues de dos undo:", p.current_value("S01", "temperature"))
p.redo()
print("redo 1:", p.current_value("S01", "temperature"))
p.redo()
print("redo 2:", p.current_value("S01", "temperature"))

try:
    p.redo()
except Empty as e:
    print("Empty:", e)

despues de dos undo: 20
redo 1: 25
redo 2: 30
Empty: No hay cambios para rehacer


### 6.3 Un cambio nuevo anula el `redo()` pendiente

In [23]:
p = DataProcessor()
p.add(("S01", "temperature", 20))
p.process_next()
p.undo()

p.add(("S01", "humidity", 60))
p.process_next()

try:
    p.redo()
except Empty as e:
    print("Empty:", e)

Empty: No hay cambios para rehacer


In [24]:
from Proyecto1.evaluador import *

In [25]:
ejecutar(DataProcessor, estudiante ='Johan Sebastian Camacho Tuay')

EVALUACION - Data Stream Processor
Estudiante : Johan Sebastian Camacho Tuay
Fecha      : 2026-09-21 19:52:43
Python     : 3.12.14 (Linux)
Evaluador  : v1.1
Clase      : DataProcessor (modulo __main__)

== Interfaz y restricciones ==

[PASS ] Define add, process_next, undo, pending y current_value
[PASS ] Se puede crear DataProcessor() sin argumentos
[PASS ] __init__ crea una ArrayQueue, un ArrayStack y una list
[PASS ] No sustituye Queue/Stack por deque u otra estructura
[PASS ] ArrayQueue y ArrayStack son las del curso (no reimplementadas)

   Interfaz y restricciones: 5/5 OK

== Pruebas obligatorias (1-17) ==

[PASS ] 1. pending() sobre un procesador vacio
[PASS ] 2. Agregar un registro
[PASS ] 3. Agregar varios registros
[PASS ] 4. Procesamiento FIFO
[PASS ] 5. Procesar un registro
[PASS ] 6. Procesar varios registros
[PASS ] 7. Actualizar una variable existente
[PASS ] 8. Consultar el valor actual
[PASS ] 9. Realizar un undo()
[PASS ] 10. Varios undo() consecutivos
[PASS ] 11. pro